# 🌊 Feature Extraction from Low-Light Underwater Images
## Interactive Exploration Notebook

This notebook walks through the full pipeline:
1. Preprocessing pipeline demo
2. Model architecture exploration
3. Training curve analysis
4. Inference and visualisation
5. Metric comparison

In [ ]:
import sys, os
sys.path.insert(0, '..')   # point to project root

import numpy as np
import torch
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Preprocessing Pipeline Demo

In [ ]:
import cv2
from src.data.preprocessing import (
    gray_world_white_balance,
    histogram_stretch_rgb,
    clahe_per_channel,
    remove_backscatter,
    estimate_illumination_map,
    underwater_color_correction,
)

# Create a synthetic degraded underwater image for demo
def make_demo_image(size=256):
    """Create synthetic blue-green underwater-like image."""
    img = np.random.randint(20, 120, (size, size, 3), dtype=np.uint8)
    # Boost blue and green, suppress red (typical underwater colour cast)
    img[:, :, 0] = (img[:, :, 0] * 0.3).astype(np.uint8)   # R weak
    img[:, :, 1] = (img[:, :, 1] * 0.7).astype(np.uint8)   # G medium
    img[:, :, 2] = (img[:, :, 2] * 1.1).clip(0, 255).astype(np.uint8)  # B strong
    # Add haze
    haze = np.random.randint(0, 40, (size, size, 3), dtype=np.uint8)
    img  = np.clip(img.astype(np.int32) + haze, 0, 255).astype(np.uint8)
    return img

demo = make_demo_image()

steps = [
    ('Original (Simulated Underwater)', demo),
    ('Histogram Stretch',      histogram_stretch_rgb(demo)),
    ('White Balance',          gray_world_white_balance(demo)),
    ('CLAHE',                  clahe_per_channel(demo)),
    ('Full Correction',        underwater_color_correction(demo)),
]

fig, axes = plt.subplots(1, len(steps), figsize=(20, 4))
fig.patch.set_facecolor('#1a1a2e')
for ax, (title, img) in zip(axes, steps):
    ax.imshow(img)
    ax.set_title(title, color='white', fontsize=9)
    ax.axis('off')
plt.suptitle('Preprocessing Pipeline', color='white', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Model Architecture Overview

In [ ]:
from src.models.feature_extractor import UnderwaterFeatureExtractor
from src.utils.utils import model_summary

model = UnderwaterFeatureExtractor(
    backbone='resnet34',
    pretrained=False,  # set True to use ImageNet weights
    num_classes=0,     # no classifier
    base_features=32,
).to(device)

model_summary(model)

# Test forward pass
x = torch.randn(1, 3, 256, 256).to(device)
with torch.no_grad():
    out = model(x)

print('\nOutput shapes:')
for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k:<15}: {tuple(v.shape)}')
    elif isinstance(v, list):
        print(f'  {k:<15}: {[tuple(t.shape) for t in v]}')

## 3. Visualise All Intermediate Maps

In [ ]:
import matplotlib.gridspec as gridspec

def tensor_to_display(t, denorm=True):
    if t.ndim == 4: t = t[0]
    t = t.detach().cpu().float()
    if t.shape[0] == 3 and denorm:
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
        t = (t * std + mean).clamp(0,1)
        return t.permute(1,2,0).numpy()
    elif t.shape[0] == 1:
        return t.squeeze(0).clamp(0,1).numpy()
    else:
        t = t.mean(0).clamp(0,1)
        t = (t - t.min()) / (t.max() - t.min() + 1e-8)
        return t.numpy()

with torch.no_grad():
    out = model(x)

items = [
    ('Input (Degraded)',      x,                   None),
    ('Enhanced Output',       out['enhanced'],      None),
    ('Noise Map',             out['noise_map'],     'hot'),
    ('Illumination Map',      out['illum_map'],     'inferno'),
    ('Attention Stage 1',     out['stage_feats'][0], 'viridis'),
    ('Attention Stage 2',     out['stage_feats'][1], 'viridis'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.patch.set_facecolor('#0d0d1a')

for idx, (title, tensor, cmap) in enumerate(items):
    ax  = axes[idx // 3][idx % 3]
    img = tensor_to_display(tensor)
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, color='white', fontsize=11)
    ax.axis('off')

plt.suptitle('Model Intermediate Outputs', color='white', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Quick Training Demo (Dummy Data)

In [ ]:
from src.losses.losses import UnderwaterCombinedLoss
from src.utils.utils import load_config

config = load_config('../config/config.yaml')
config['training']['epochs'] = 5

# Shrink for demo
config['model']['base_features'] = 16
config['model']['backbone'] = 'resnet18'

criterion = UnderwaterCombinedLoss(config).to(device)
model_demo = UnderwaterFeatureExtractor(
    backbone='resnet18', pretrained=False,
    base_features=16, num_classes=0
).to(device)

optim = torch.optim.AdamW(model_demo.parameters(), lr=1e-4)
loss_history = []

print('Mini training loop (5 steps):')
for step in range(5):
    x   = torch.rand(2, 3, 128, 128).to(device)
    ref = torch.rand(2, 3, 128, 128).to(device)
    
    optim.zero_grad()
    out = model_demo(x)
    out['input'] = x
    loss, loss_dict = criterion(out, ref)
    loss.backward()
    optim.step()
    
    loss_history.append(loss.item())
    print(f'  Step {step+1}: loss={loss.item():.4f}')

plt.figure(figsize=(6, 3))
plt.plot(loss_history, 'o-', color='#e94560')
plt.title('Training Loss', color='black')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Evaluation Metrics on Synthetic Data

In [ ]:
from src.evaluation.metrics import compute_uiqm, compute_uciqe, compute_psnr, compute_ssim

# Simulate degraded vs enhanced
degraded_np  = make_demo_image(256)
enhanced_np  = underwater_color_correction(degraded_np)

uiqm_deg  = compute_uiqm(degraded_np)
uiqm_enh  = compute_uiqm(enhanced_np)
uciqe_deg = compute_uciqe(degraded_np)
uciqe_enh = compute_uciqe(enhanced_np)

print('No-Reference Metrics:')
print(f'  UIQM  — Degraded: {uiqm_deg:.4f}  |  Enhanced: {uiqm_enh:.4f}  |  Δ={uiqm_enh-uiqm_deg:+.4f}')
print(f'  UCIQE — Degraded: {uciqe_deg:.4f}  |  Enhanced: {uciqe_enh:.4f}  |  Δ={uciqe_enh-uciqe_deg:+.4f}')

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(degraded_np);  axes[0].set_title(f'Degraded\nUIQM={uiqm_deg:.3f}  UCIQE={uciqe_deg:.3f}')
axes[1].imshow(enhanced_np);  axes[1].set_title(f'Enhanced\nUIQM={uiqm_enh:.3f}  UCIQE={uciqe_enh:.3f}')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()